# 01 · Exploration on a single frame

Question: how to separate the lecturer from the board? Approaches compared on the same frame:

1. **DeepLab V3 on the full frame** (what the MediaPipe web demo does)
2. **DeepLab V3 on a crop** around the lecturer
3. **PoseLandmarker** (detects the person, crops, segments; it also gives the landmarks, hence the pointed spot)
4. **Geometry**: board quadrilateral, person pixels ∩ board
5. **Colour**: how separable skin, slate and chalk are

The frame is the censored one (lecturer replaced by areas of average colour): the figures about the person
must be confirmed on a real frame. Models come from the catalog by name (`vision-toolkit models list`).

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt

from vision_toolkit.core.geometry import polygon_mask, rectify_homography
from vision_toolkit.models import catalog
from vision_toolkit.usecases.board.detect import detect_board
from vision_toolkit.usecases.board.person_mask import PersonMasker

ROOT = Path.cwd().parent
VIDEO_CROP = (0, 68, 1674, 783)  # x, y, w, h: video area in the player screenshot
x, y, w, h = VIDEO_CROP
frame = cv2.imread(str(ROOT / "data/frame/lezione_prova_cit_censured.png"))[y:y+h, x:x+w]

def show(*imgs, titles=None, size=6):
    fig, axs = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size * 0.6))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(axs), imgs)):
        ax.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
        ax.set_title(titles[i] if titles else ""); ax.axis("off")
    plt.tight_layout(); plt.show()

show(frame, titles=["frame (video area)"], size=10)

## 1–2 · DeepLab V3: full frame vs crop

In [ ]:
deeplab = catalog.create("deeplab-v3", confidence=True)
person_conf = lambda img: deeplab(img).confidence_of("person")

full = person_conf(frame)
crop = frame[292:712, 330:760]
cropped = person_conf(crop)
print(f"full frame: max confidence {full.max():.3f}, pixels > 0.5: {(full > 0.5).sum()}")
print(f"crop:       max confidence {cropped.max():.3f}, pixels > 0.5: {(cropped > 0.5).sum()}")
show(full, crop, cropped, titles=["person, full frame", "crop", "person, crop"])

The model runs at 257×257: in the full frame the lecturer shrinks to a couple of dozen pixels and disappears.
The problem is **resolution**, not colour. Averaging colours per region helped in the demo
because it made the parts more uniform, but it needs the mask to be known already: it is circular.
`ops.crop_refine.segment_on_crop` turns this lesson into an op that works with any segmenter.

## 3 · PoseLandmarker: mask and pointed spot

In [ ]:
with PersonMasker() as masker:
    person = masker(frame)
print("index fingertip:", person.pointer, " arm-torso angle:", round(person.arm_angle), "degrees")
vis = frame.copy()
vis[person.mask > 0] = (vis[person.mask > 0] * 0.4 + np.array([255, 0, 255]) * 0.6).astype(np.uint8)
cv2.circle(vis, person.pointer, 10, (0, 255, 0), -1)
show(vis, titles=["pose mask + arm skeleton (dilated), fingertip in green"], size=10)

It works on the full frame because the internal detector crops the person before segmenting it.
The pose mask alone lost a stretch of the raised arm: `PersonMasker` covers it by drawing the arms
as thick segments between shoulder, elbow, wrist and index fingertip (`ops.skeleton`).
`PersonMasker(refine_model="deeplab-v3")` adds DeepLab on the crop.

## 4 · Geometry: board and occluded pixels

In [ ]:
quad_raw = detect_board(frame)
quad = detect_board(frame, person.mask)
print("without person mask:", None if quad_raw is None else quad_raw.astype(int).tolist())
print("with person mask:   ", quad.astype(int).tolist())
bm = polygon_mask(frame.shape, quad)
occ = (bm > 0) & (person.mask > 0)
print(f"occluded board pixels: {occ.sum()} of {(bm > 0).sum()} ({occ.sum() / (bm > 0).sum():.1%})")

H, size = rectify_homography(quad)
vis = frame.copy(); cv2.polylines(vis, [quad.astype(np.int32)], True, (0, 0, 255), 3)
vis[occ] = (vis[occ] * 0.4 + np.array([255, 0, 255]) * 0.6).astype(np.uint8)
show(vis, cv2.warpPerspective(frame, H, size), titles=["board and occlusion", "rectified board"])

The legs below the board distort the contour unless the person is removed first.
On a video the simplest solution is to look for the board on the **temporal median** of the frames,
where the lecturer disappears (`core.io.median_frame`); otherwise the 4 corners are set by hand once per room.

## 5 · Colour: skin, slate, chalk

In [ ]:
arm = cv2.imread(str(ROOT / "data/calibration/sample_arm.png"), cv2.IMREAD_UNCHANGED)
arm_px = arm[arm[..., 3] > 200][:, :3]
to_lab = lambda px: cv2.cvtColor(px.reshape(-1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(float)

board_px = to_lab(frame[(bm > 0) & (person.mask == 0)])
L = board_px[:, 0]
groups = {"arm": to_lab(arm_px),
          "slate": board_px[L < np.percentile(L, 50)],
          "chalk": board_px[L > np.percentile(L, 97)]}
for name, px in groups.items():
    print(f"{name:8s} Lab median {np.median(px, 0).round(0)}  std {px.std(0).round(1)}")

fig, ax = plt.subplots(figsize=(6, 5))
for name, px in groups.items():
    s = px[np.random.default_rng(0).choice(len(px), min(3000, len(px)), replace=False)]
    ax.scatter(s[:, 1] - 128, s[:, 0], s=2, alpha=0.4, label=name)
ax.set_xlabel("a (green - red)"); ax.set_ylabel("L (lightness, 0-255 scale)"); ax.legend(); plt.show()
deeplab.close()

Slate is very compact, so "not board" is easy to recognise. But arm and chalk have the same
lightness and differ little in colour; a white shirt is indistinguishable from chalk.
Colour alone is not enough: it needs shape (chalk strokes are thin) or the person model.
Useful to refine the edges of the mask, not as the main criterion.

## Conclusion

On a single frame: **PoseLandmarker + arm skeleton** for the person, **geometry** for the board.
The decisive step, though, is temporal: the board stays still and the lecturer moves, so every pixel
covered now was visible shortly before or will be shortly after: `02_temporal_reconstruction.ipynb`.